# Scenario check (T08)

Re-runs the code shown in every demo scenario (`web/lib/mock/scenarios/sc_*.json`) on the generated synthetic datasets (`web/lib/mock/datasets/`) and confirms that what the scenario shows is what the code produces.

For each scenario it checks that:

1. each dataset's profile JSON matches the file (rows, duplicates, nulls, unique values);
2. every `code` event runs, in order, in one shared namespace with `df` = the raw file as pandas reads it with default options;
3. the `stdout` lines match the real output (trailing spaces ignored);
4. failed attempts fail with the same `error_type` and first message line, and successful ones do not fail;
5. every artifact value (metric, table cells, chart x/y) appears in that step's real output.

Mock output is scripted. This notebook only proves the scripts are consistent with the code and data; it measures nothing about the agent.

Run: `pip install pandas openpyxl jupyter`, then `jupyter nbconvert --to notebook --execute --inplace docs/scenario-checks/check_scenarios.ipynb`. If the data or a scenario changes, fix the scenario (or rerun `npm run mock:generate` and rebuild it) until this passes.

In [1]:
import contextlib, copy, io, json, re
from pathlib import Path
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "web/lib/mock/scenarios").is_dir())
DATA = ROOT / "web/lib/mock/datasets"
FILES = {"ds_delivery": "project_delivery.csv", "ds_sales": "regional_sales.xlsx", "ds_tickets": "support_tickets.csv",
         "ds_costs": "project_costs.csv", "ds_hr": "team_allocation.csv"}
SCENARIOS = {p.stem: json.loads(p.read_text(encoding="utf8")) for p in sorted((ROOT / "web/lib/mock/scenarios").glob("sc_*.json"))}

def load(ds, **kw):
    path = DATA / FILES[ds]
    return pd.read_excel(path, **kw) if path.suffix == ".xlsx" else pd.read_csv(path, **kw)

print("pandas", pd.__version__, "|", len(SCENARIOS), "scenarios:", ", ".join(SCENARIOS))

pandas 3.0.6 | 5 scenarios: sc_01, sc_02, sc_03, sc_04, sc_05


## 1. Dataset profiles match the files

In [2]:
def check_profile(ds):
    prof = json.loads((DATA / f"{ds}.profile.json").read_text(encoding="utf8"))
    df = load(ds, header=1 if ds == "ds_costs" else 0)  # ds_costs has a title row above the header
    out = []
    if prof["rows"] != len(df): out.append(f"rows {prof['rows']} != {len(df)}")
    if prof["duplicates"] != int(df.duplicated().sum()): out.append(f"duplicates {prof['duplicates']} != {int(df.duplicated().sum())}")
    if [c["name"] for c in prof["columns"]] != list(df.columns): out.append("column names differ")
    for c in prof["columns"]:
        if c["name"] not in df: continue
        s = df[c["name"]]
        if c["nulls"] != int(s.isna().sum()): out.append(f"{c['name']}: nulls {c['nulls']} != {int(s.isna().sum())}")
        if c["unique"] != int(s.nunique()): out.append(f"{c['name']}: unique {c['unique']} != {int(s.nunique())}")
    return out

profile_problems = {ds: check_profile(ds) for ds in FILES}
for ds, p in profile_problems.items(): print(ds, "OK" if not p else p)
assert not any(profile_problems.values())

ds_delivery OK
ds_sales OK
ds_tickets OK
ds_costs OK
ds_hr OK


## 2. Replay every scenario

In [3]:
NUM = re.compile(r"-?\d[\d,]*(?:\.\d+)?")
numbers = lambda text: {float(m.replace(",", "")) for m in NUM.findall(text)}

def artifact_values(a):
    p = a["payload"]
    if a["kind"] == "metric": return [p["value"]]
    if a["kind"] == "table": return [c for row in p["rows"] for c in row]
    return [*p["data"][0]["x"], *p["data"][0]["y"]]

def traced(value, text, nums):
    if value is None or isinstance(value, bool): return True
    if isinstance(value, (int, float)): return float(value) in nums
    return str(value) in text

def attempts_of(sc):
    attempts, cur = [], None
    for e in sc["events"]:
        if e["type"] == "step_started":
            cur = dict(step=e["step_id"], attempt=e["attempt"], code=None, out=[], error=None, artifacts=[])
            attempts.append(cur)
        elif cur is None: continue
        elif e["type"] == "code": cur["code"] = e["source"]
        elif e["type"] == "stdout": cur["out"].append(e["text"])
        elif e["type"] == "error": cur["error"] = e
        elif e["type"] == "artifact": cur["artifacts"].append(e)
    return attempts

def replay(sc):
    problems, stats = [], dict(attempts=0, stdout_lines=0, artifacts=0)
    ns = {"pd": pd, "df": load(sc["dataset_id"])}
    for a in attempts_of(sc):
        label = f"{sc['scenario_id']} {a['step']} attempt {a['attempt']}"
        buf, exc = io.StringIO(), None
        try:
            with contextlib.redirect_stdout(buf): exec(a["code"], ns)
        except Exception as ex:
            exc = ex
        actual = [l.rstrip() for l in buf.getvalue().splitlines() if l.strip()]
        stats["attempts"] += 1; stats["stdout_lines"] += len(actual)
        if actual != a["out"]:
            problems.append(f"{label}: stdout differs\n  shown : {a['out']}\n  actual: {actual}")
        if a["error"] is None and exc is not None:
            problems.append(f"{label}: raised {type(exc).__name__}: {exc}")
        if a["error"] is not None:
            if exc is None: problems.append(f"{label}: expected {a['error']['error_type']} but it ran fine")
            else:
                first = lambda m: str(m).splitlines()[0] if str(m) else ""
                if type(exc).__name__ != a["error"]["error_type"] or first(exc) != first(a["error"]["message"]):
                    problems.append(f"{label}: error differs\n  shown : {a['error']['error_type']}: {first(a['error']['message'])}\n  actual: {type(exc).__name__}: {first(exc)}")
        text, nums = "\n".join(actual), numbers("\n".join(actual))
        for art in a["artifacts"]:
            stats["artifacts"] += 1
            missing = [v for v in artifact_values(art) if not traced(v, text, nums)]
            if missing: problems.append(f"{label}: artifact '{art['title']}' has values not in the step output: {missing}")
    return problems, stats

In [4]:
rows = []
for sid, sc in SCENARIOS.items():
    problems, stats = replay(sc)
    rows.append(dict(scenario=sid, dataset=sc["dataset_id"], **stats, problems=len(problems)))
    for p in problems: print(p)
result = pd.DataFrame(rows)
print(result.to_string(index=False))
assert result["problems"].sum() == 0, "fix the scenarios above"
print("\nAll scenario outputs match the code and data.")

scenario     dataset  attempts  stdout_lines  artifacts  problems
   sc_01 ds_delivery         4            25          5         0
   sc_02    ds_sales         5            25          3         0
   sc_03  ds_tickets         3            18          2         0
   sc_04    ds_costs         3            24          2         0
   sc_05       ds_hr         3             3          0         0



All scenario outputs match the code and data.


## 3. The check can fail

A scenario with one wrong number must be flagged, otherwise a pass means nothing.

In [5]:
bad = copy.deepcopy(SCENARIOS["sc_01"])
line = next(e for e in bad["events"] if e["type"] == "stdout" and e["text"].startswith("Removed"))
line["text"] = "Removed 15 duplicate rows, 465 tasks left"
found, _ = replay(bad)
assert found, "checker missed a wrong number"
print("Wrong number caught:", found[0].splitlines()[0])

Wrong number caught: sc_01 s1 attempt 1: stdout differs
